# End-to-End BO：逐次 → 同時Batch → 非同期の実験ワークフロー

同じ `SingleTaskGP` を使い、**1点ずつ測定する逐次BO**、**2点を一度に選ぶjoint Batch BO**、**未完了の実験を考慮する非同期BO**を順番に体験します。

どの方法でも共通するのは、**完了した観測だけでGPを学習する**ことです。候補を選んだだけの状態、実験中の状態、測定値が返った状態を区別します。独自のBOフレームワークや獲得関数は作成しません。


## 3つの実験方式を区別する

| 方式 | 新しく提案する点 | 未完了の実験 |
|---|---|---|
| 逐次BO | `q=1` | 原則、測定完了後に次を選ぶ |
| Joint Batch BO | `q=2` | 同時に選んだ2点を測定する |
| 非同期BO | `q=1` など | `X_pending` に未完了点を渡す |

**`q=2` は `q=1` を2回独立に呼ぶことではありません。** また、`X_pending` は未完了の実験入力であり、教師データではありません。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import (
    qLogExpectedImprovement,
    qLogNoisyExpectedImprovement,
)
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 共通のGPと獲得関数

目的関数は1次元・単一目的の最大化です。`qLogExpectedImprovement` は観測済み最良値に対する改善を評価し、`qLogNoisyExpectedImprovement` はベースライン観測とpending点を考慮します。

このNotebookでは実験方式の違いを見やすくするため、すべて同じ目的関数と探索範囲 `[0,1]` を使います。

## 2. Problem and completed observations

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.25 * X


bounds = torch.tensor([[0.0], [1.0]])
train_X = torch.linspace(0.05, 0.95, 8).unsqueeze(-1)
train_Y = objective(train_X)

print("completed observations:", train_X.shape[0])

### 初期観測だけがGPの学習データになる

`train_X/train_Y` には完了済みの8件を入れます。実験装置へ候補を送っただけでは学習データは増えません。

逐次・Batch・非同期の各方式で、観測が完了したときだけ新しい `Y` を追加します。

## 3. A small public-API fitting function

Workflow内の重複を減らすための局所helperです。robotorchan専用frameworkを新設する意図はありません。

In [ ]:
def fit_model(X: torch.Tensor, Y: torch.Tensor) -> SingleTaskGP:
    model = SingleTaskGP(X, Y)
    model.fit()
    model.eval()
    return model

### `fit()` は既存モデルの公開APIを使用する

局所的な `fit_model()` は、モデル構築・`fit()`・評価モードへの切り替えをまとめるだけの小さな関数です。学習ロジックや獲得関数は再実装しません。

`SingleTaskGP.fit()` を使用し、Notebook内でMLLの最適化を直接記述しないようにします。

## 4. Sequential BO: fit → acquisition → candidate → evaluate → append

### 逐次BO：1点測定するたびに更新

各反復でGPを再学習し、`q=1` で1点を提案します。目的関数を評価して `Y` が得られた後、初めて訓練データへ追加します。

逐次BOは次の候補を選ぶ時点で**直前の観測結果を利用できる**のが特徴です。

In [ ]:
for iteration in range(2):
    model = fit_model(train_X, train_Y)
    acquisition = qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
        sampler=SobolQMCNormalSampler(torch.Size([64]), seed=100 + iteration),
    )
    candidate, _ = optimize_acqf(
        acq_function=acquisition,
        bounds=bounds,
        q=1,
        optimizer="botorch",
        num_restarts=3,
        raw_samples=64,
    )
    candidate = candidate.detach()
    observed_Y = objective(candidate)
    train_X = torch.cat([train_X, candidate], dim=0)
    train_Y = torch.cat([train_Y, observed_Y], dim=0)
    print(iteration, candidate.item(), observed_Y.item())

### 逐次BOの履歴を可視化する

初期観測と逐次BOで追加した観測を比較します。最良観測値はGP予測の最大値ではなく、**実際に測定したYの最大値**です。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_X.squeeze(-1), objective(plot_X).squeeze(-1), label="True objective")
ax.scatter(train_X[:8, 0], train_Y[:8, 0], s=25, label="Initial observations")
ax.scatter(train_X[8:, 0], train_Y[8:, 0], marker="x", s=70, label="Sequential observations")
ax.set(xlabel="x", ylabel="objective", title="Sequential BO observations")
ax.legend()
plt.show()

## 5. Joint batch BO: q is a joint decision

`q=2`はq=1 optimizerを2回呼ぶことと同じではありません。MC acquisitionは2候補のjoint posterior utilityを評価します。

### Joint q=2の意味

2候補を**同時に最適化**するため、MC獲得関数は候補間の予測相関も含めて2点の改善を評価します。

`batch_X` のshapeは `[2,1]` です。これは「2候補 × 1入力変数」であり、2出力モデルという意味ではありません。

In [ ]:
model = fit_model(train_X, train_Y)
batch_acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=201),
)
batch_X, _ = optimize_acqf(
    acq_function=batch_acquisition,
    bounds=bounds,
    q=2,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=64,
)
batch_X = batch_X.detach()
batch_Y = objective(batch_X)

assert batch_X.shape == torch.Size([2, 1])
train_X = torch.cat([train_X, batch_X], dim=0)
train_Y = torch.cat([train_Y, batch_Y], dim=0)
print("joint batch:\n", batch_X)

### Batch BOで追加した2点を確認する

2点とも測定が完了したと仮定してから訓練データへ追加します。非同期BOとは異なり、この例では**Batchの途中で次の候補を提案しません**。

In [ ]:
print("Joint batch shape:", batch_X.shape)
print("Batch responses:", batch_Y.squeeze(-1))
print("Completed observations after batch:", train_X.shape[0])
assert train_X.shape[0] == 8 + 2 + 2

## 6. Async BO: completed data and pending work are different states

2 workersが実行中で、その間にreplacement candidateを1点生成する状況を作ります。pending pointsはtraining dataへ追加しません。

### 非同期BO：2台の装置がまだ実験中

`X_pending` の2点は実験装置に投入済みですが、まだ応答が返っていません。GPの訓練データには追加せず、獲得関数に**未完了実験の入力として渡します**。

空いた装置に次に投入する候補を1点提案するため、最適化の `q` は1です。

In [ ]:
X_pending = torch.tensor([[0.22], [0.78]])
completed_rows = train_X.shape[0]
model = fit_model(train_X, train_Y)
async_acquisition = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    X_pending=X_pending,
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=301),
    prune_baseline=False,
)
replacement_X, _ = optimize_acqf(
    acq_function=async_acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=64,
)

assert train_X.shape[0] == completed_rows
assert replacement_X.shape == torch.Size([1, 1])
print("pending workers:\n", X_pending)
print("replacement candidate:", replacement_X.detach())

### 非同期の状態を図で確認する

完了済みの観測、実験中の2点、追加提案した1点を別々に表示します。pending点と新しい提案点にはまだ測定値がないため、**縦軸の実測値としては表示しません**。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_X.squeeze(-1), objective(plot_X).squeeze(-1), label="True objective (demo only)")
ax.scatter(train_X[:, 0], train_Y[:, 0], s=22, label="Completed observations")
for x in X_pending.squeeze(-1):
    ax.axvline(x.item(), linestyle="--", color="tab:orange", alpha=0.7)
ax.axvline(replacement_X.detach().item(), linestyle=":", color="tab:red", label="Proposed replacement")
ax.set(xlabel="x", ylabel="objective", title="Async: completed vs pending vs proposed")
ax.legend()
plt.show()

## 7. Complete one worker, keep the other pending, and continue

完了した評価だけをtraining dataへ追加します。未完了点は次のacquisitionでも`X_pending`として残します。

### 1件だけ完了したらどうするか

pendingのうち1点だけが完了したと仮定し、観測結果を訓練データへ追加します。もう1点は引き続き `remaining_pending` として獲得関数へ渡します。

この例では先ほどの `replacement_X` は**提案しただけで未投入**として扱います。実際に実験を開始した場合は、その入力もpending集合へ追加して管理してください。

In [ ]:
completed_X = X_pending[:1]
remaining_pending = X_pending[1:]
completed_Y = objective(completed_X)
train_X = torch.cat([train_X, completed_X], dim=0)
train_Y = torch.cat([train_Y, completed_Y], dim=0)

model = fit_model(train_X, train_Y)
next_acquisition = qLogNoisyExpectedImprovement(
    model=model,
    X_baseline=train_X,
    X_pending=remaining_pending,
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=302),
    prune_baseline=False,
)
next_X, _ = optimize_acqf(
    acq_function=next_acquisition,
    bounds=bounds,
    q=1,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=64,
)

print("new completed rows:", train_X.shape[0])
print("still pending:", remaining_pending)
print("next candidate:", next_X.detach())

### 更新後のGP予測を可視化する

非同期で1件の実験が完了した前後のGP事後平均と不確実性を比較します。残りのpending点は予測には影響する獲得関数の文脈ですが、**GPの教師データにはなりません**。

In [ ]:
before_async_model = fit_model(train_X[:-1], train_Y[:-1])
with torch.no_grad():
    before_async = before_async_model.posterior(plot_X)
    after_async = model.posterior(plot_X)
fig, ax = plt.subplots(figsize=(9, 4))
for post, name in ((before_async, "Before completion"), (after_async, "After completion")):
    m = post.mean.squeeze(-1)
    sd = post.variance.clamp_min(0).sqrt().squeeze(-1)
    ax.plot(plot_X.squeeze(-1), m, label=name)
    ax.fill_between(plot_X.squeeze(-1), m - 2*sd, m + 2*sd, alpha=0.12)
ax.axvline(completed_X.item(), linestyle="--", label="Completed worker")
ax.axvline(remaining_pending.item(), linestyle=":", label="Still pending")
ax.set(xlabel="x", ylabel="GP prediction", title="GP update after async completion")
ax.legend()
plt.show()

## 8. What changes across workflows?

| Workflow | Training data | Acquisition context | Candidate semantics |
| --- | --- | --- | --- |
| Sequential | completed only | none | q=1 |
| Joint batch | completed only | none | q>1 joint utility |
| Async | completed only | `X_pending` | replacement candidate(s) |

Pending points、fantasy-model batch、one-shot auxiliary rowsはすべて別概念です。

## 9. Choose the specialized workflow by problem structure

| Need | Notebook |
| --- | --- |
| first standard BO loop | `01_single_task_gp.ipynb` |
| mixed continuous/categorical | `02_mixed_single_task_gp.ipynb` |
| multi-fidelity / cost-aware | `03_multi_fidelity_gp.ipynb` |
| MultiTask / Kronecker | `04_multitask_gp.ipynb` |
| high-dimensional reduction | `19_reduced_gp.ipynb` |
| multi-objective | `33_multiobjective_acquisition.ipynb` |
| regression active learning | `34_active_learning_acquisition.ipynb` |
| objective / posterior transform | `36_objective_posterior_transform.ipynb` |
| sampling contract | `37_posterior_sampling.ipynb` |
| optimizer / constraints | `39_acquisition_optimization.ipynb` |
| batch / async / fantasy | `40_batch_async_fantasization.ipynb` |
| initialization | `38_acquisition_initialization.ipynb` |
| TuRBO | `47_turbo_trust_region.ipynb` |
| robust input perturbation | `48_robust_input_perturbation.ipynb` |

## 10. Cross-layer checklist

実運用Workflowでは、モデル名だけでなく次を確認します。

1. training dataのshapeと構造列
2. posterior / sampler contract
3. objective / posterior transform semantics
4. acquisitionがsingle, batch, noisy, MO, MF, ALのどれか
5. candidate constraintsとoutcome constraintsの区別
6. initialization shapeとone-shot例外
7. optimizerがvariable typeとq semanticsを扱えるか
8. pending / fantasy lifecycle
9. candidateがどの座標系で返るか
10. evaluation後に何をtraining dataとstateへ戻すか

## 11. Related documentation

- `docs-internal/architecture/notebook-executable-documentation-architecture.md`
- `docs-internal/maintenance/notebook-authoring-standard.md`
- `docs-internal/contracts/optimization/acquisition-integration.md`
- `docs-internal/contracts/optimization/batch-async-fantasization.md`

このNotebookを入口にし、問題構造が増えた時だけ専門Workflowへ分岐するのが推奨構成です。

## 12. 実務での選択基準

| 条件 | 推奨する方式 | 既存API |
|---|---|---|
| 実験が1件ずつ完了し、その都度更新できる | 逐次BO | `qLogExpectedImprovement`, `q=1` |
| 複数点を同時に決定して測定する | Joint Batch BO | `qLogExpectedImprovement`, `q=2` |
| 実験時間が異なり、空いた装置から次を開始する | 非同期BO | `qLogNoisyExpectedImprovement(X_pending=...)` |

**注意：** `X_pending` は実験が始まった時点で登録し、実験完了時にはpendingから外して観測値を訓練データへ追加します。提案だけの候補を勝手に観測済みとして扱わないでください。